# Assignment 3: Bug Fixing — RAG & Zero-shot
**CSCI 455/555 — Spring 2026 | Prof. Antonio Mastropaolo**

Notebook **3 of 3**. A paradigm-level comparison: task-specific small model (Notebook 2) vs. prompt-a-large-model-with-context.

**Generator**: `Qwen/Qwen2.5-Coder-1.5B-Instruct` (~25× larger than our T5).

**Retriever** (design choice — justified below): `microsoft/codebert-base` encoder with mean-pooling over hidden states → 768-dim dense vector → FAISS `IndexFlatL2`.

*Why CodeBERT mean-pooling over Word2Vec / Code2Vec?*
- CodeBERT is pre-trained on 6 programming languages (Java included) with MLM+RTD — it produces *contextual* embeddings that capture both identifier semantics and surrounding code structure.
- Word2Vec (RAG lab) produces one static vector per token regardless of context; it struggles when the same token (e.g., `i`) means different things in different loops.
- No AST parsing needed (unlike Code2Vec), which keeps the retriever robust to malformed buggy code — the input domain here.

**Evaluation** runs two settings on the same test split used in Notebook 2 and compares all four configurations in a final table.

## 0.1) Install Dependencies

In [4]:
!pip install -q transformers==4.46.0 tokenizers==0.20.3 sentencepiece==0.2.0
!pip install -q "datasets>=2.21.0" torch tqdm matplotlib
!pip install -q faiss-cpu accelerate
from IPython.display import clear_output; clear_output(); print("Installations Complete")

Installations Complete


## 0.2) Imports

In [5]:
import os
import json
import pickle
import random
import numpy as np
import torch
import faiss
from tqdm import tqdm

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForCausalLM,
)
from datasets import load_dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cpu


## 0.3) Configuration

In [6]:
# =======================================================================
# Paths
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = r"/content/drive/MyDrive/W&M/GenAI/Assignment_3"
KB_DIR      = f"{BASE_DIR}/rag_kb"
RESULTS_DIR = f"{BASE_DIR}/results"

os.makedirs(KB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# =======================================================================
# Models
RETRIEVER_MODEL = "microsoft/codebert-base"
GENERATOR_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# =======================================================================
# Retriever
EMBED_MAX_LEN = 256
EMBED_BATCH   = 32
TOP_K         = 3

# =======================================================================
# Generation
MAX_NEW_TOKENS   = 256
TEMPERATURE      = 0.0   # greedy (bug fixing prefers absolute best answer, not random)
GEN_MAX_CTX_LEN  = 4096

# =======================================================================
# Evaluation — match Notebook 2 so results are comparable
TEST_SUBSET_SIZE = 500  # None for full test set

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
print("✓ Configuration loaded")

Mounted at /content/drive
✓ Configuration loaded


---
---

# Task 1: Build the Knowledge Base

KB = the 52K (buggy, fixed) pairs from the fine-tuning TRAINING split. We embed each **buggy** method (not the fix) — at inference time the query is also a buggy method, so query space and index space match.

---
---

## 1.1) Load Bug-Fix Training Pairs

In [5]:
from datasets import load_dataset

ds_train = load_dataset("google/code_x_glue_cc_code_refinement", "medium",
                        split="train", streaming=True, trust_remote_code=True)
ds_test  = load_dataset("google/code_x_glue_cc_code_refinement", "medium",
                        split="test",  streaming=True, trust_remote_code=True)

train_pairs = [{"buggy": r["buggy"], "fixed": r["fixed"]} for r in ds_train]
test_pairs  = [{"buggy": r["buggy"], "fixed": r["fixed"]} for r in ds_test]

print(f"Knowledge base : {len(train_pairs):,} pairs")
print(f"Test set       : {len(test_pairs):,} instances")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/code_x_glue_cc_code_refinement' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/code_x_glue_cc_code_refinement' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restar

README.md: 0.00B [00:00, ?B/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/code_x_glue_cc_code_refinement' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/code_x_glue_cc_code_refinement' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Knowledge base : 52,364 pairs
Test set       : 6,545 instances


## 1.2) CodeBERT Encoder with Mean-Pooling

Given a batch of code strings, produce one 768-dim vector each by averaging last-layer hidden states over non-padding positions.

In [6]:
cb_tok = AutoTokenizer.from_pretrained(RETRIEVER_MODEL)
cb_model = AutoModel.from_pretrained(RETRIEVER_MODEL).to(DEVICE)
cb_model.eval()

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.no_grad()
def encode_codebert(texts, batch_size=EMBED_BATCH, max_length=EMBED_MAX_LEN):
    out = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding", disable=True):
        chunk = texts[start:start + batch_size]
        enc = cb_tok(chunk, padding=True, truncation=True, max_length=max_length,
                     return_tensors="pt").to(DEVICE)
        h = cb_model(**enc).last_hidden_state
        v = mean_pool(h, enc["attention_mask"]).cpu().numpy().astype("float32")
        out.append(v)
    return np.concatenate(out, axis=0)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

## 1.3) Embed the KB and Build the FAISS Index

In [7]:
buggy_texts = [p["buggy"] for p in train_pairs]
kb_embeddings = encode_codebert(buggy_texts)
print(f"KB embeddings shape: {kb_embeddings.shape}")

dim = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(kb_embeddings)
print(f"FAISS index built: {index.ntotal:,} vectors, dim={dim}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

KB embeddings shape: (52364, 768)
FAISS index built: 52,364 vectors, dim=768


## 1.4) Save the KB

In [8]:
faiss.write_index(index, f"{KB_DIR}/index.faiss")
with open(f"{KB_DIR}/pairs.pkl", "wb") as f:
    pickle.dump(train_pairs, f)
with open(f"{KB_DIR}/config.json", "w") as f:
    json.dump({"retriever": RETRIEVER_MODEL, "dim": int(dim),
               "n_pairs": len(train_pairs)}, f, indent=2)
print(f"✓ KB saved → {KB_DIR}")

✓ KB saved → /content/drive/MyDrive/W&M/GenAI/Assignment_3/rag_kb


---
---

# Task 2: Retriever

---
---

In [9]:
class BugFixRetriever:
    def __init__(self, kb_dir=KB_DIR):
        self.index = faiss.read_index(f"{kb_dir}/index.faiss")
        with open(f"{kb_dir}/pairs.pkl", "rb") as f:
            self.pairs = pickle.load(f)
        with open(f"{kb_dir}/config.json") as f:
            self.config = json.load(f)
        print(f"Loaded KB: {self.config['n_pairs']:,} pairs, dim={self.config['dim']}")

    def retrieve(self, query, top_k=TOP_K):
        q_vec = encode_codebert([query])
        dists, idxs = self.index.search(q_vec, top_k)
        hits = []
        for d, i in zip(dists[0], idxs[0]):
            if 0 <= i < len(self.pairs):
                p = self.pairs[i]
                hits.append({"buggy": p["buggy"], "fixed": p["fixed"],
                             "similarity": 1.0 / (1.0 + float(d))})
        return hits

retriever = BugFixRetriever()

Loaded KB: 52,364 pairs, dim=768


## 2.1) Test Retrieval

In [10]:
query = test_pairs[0]["buggy"]
print(f"QUERY:\n{query[:200]}\n")

hits = retriever.retrieve(query, top_k=TOP_K)
for i, h in enumerate(hits, 1):
    print(f"--- Hit {i}  (sim={h['similarity']:.3f}) ---")
    print(f"  BUGGY: {h['buggy'][:120]}")
    print(f"  FIXED: {h['fixed'][:120]}\n")

QUERY:
public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) ) ) { return VAR_2 . METHOD_4 ( ) ; } else if ( METHOD_3 ( VAR_3 . METHOD_5 ( ) . METHOD_6 ( ) ) ) {

--- Hit 1  (sim=0.227) ---
  BUGGY: private boolean METHOD_1 ( ) { VAR_1 = null ; if ( VAR_2 . METHOD_2 ( ( ( VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) ) != nul
  FIXED: private boolean METHOD_1 ( ) { VAR_1 = null ; if ( ( VAR_2 . METHOD_2 ( ( ( VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) ) != n

--- Hit 2  (sim=0.217) ---
  BUGGY: public TYPE_1 METHOD_1 ( ) { if ( ( this . METHOD_2 ( ) ) || ( this . METHOD_3 ( ) ) ) { return VAR_1 . METHOD_4 ( ) ; }
  FIXED: public TYPE_1 METHOD_1 ( ) { if ( ( this . METHOD_2 ( ) ) || ( this . METHOD_3 ( ) ) ) { return VAR_1 . METHOD_4 ( ) ; }

--- Hit 3  (sim=0.215) ---
  BUGGY: public java.lang.Void METHOD_1 ( ) { if ( ! ( VAR_1 . METHOD_2 ( ) . METHOD_3 ( ) . equals ( VAR_1 . METHOD_2 ( ) . getI
  FIXED: public java.lang.Void METHOD_1 ( ) { if ( ( ( VAR

---
---

# Task 3: Generator — Qwen2.5-Coder-1.5B-Instruct

---
---

In [11]:
qwen_tok = AutoTokenizer.from_pretrained(GENERATOR_MODEL, trust_remote_code=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
qwen_model.eval()

if qwen_tok.pad_token is None:
    qwen_tok.pad_token = qwen_tok.eos_token
qwen_tok.padding_side = "left"
print(f"Loaded {GENERATOR_MODEL} on {qwen_model.device}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-Coder-1.5B-Instruct on cuda:0


## 3.1) Prompt Builders — Few-shot (RAG) and Zero-shot

Both variants share the same system message and ask for a single fixed Java method as the response — the only difference is whether retrieved exemplars are included.

In [13]:
SYSTEM_MSG = (
    "You are an expert Java developer fixing bugs in methods. "
    "Given a buggy Java method, respond with ONLY the corrected method. "
    "Do not add explanations, comments, or markdown — output raw Java code."
)


def build_few_shot_prompt(query_buggy, retrieved):
    blocks = ["Here are examples of buggy Java methods and their fixes:\n"]
    for i, ex in enumerate(retrieved, 1):
        blocks.append(f"--- Example {i} ---")
        blocks.append(f"Buggy:\n{ex['buggy']}\n")
        blocks.append(f"Fixed:\n{ex['fixed']}\n")
    blocks.append("Now fix the following buggy method:\n")
    blocks.append(f"Buggy:\n{query_buggy}\n")
    blocks.append("Fixed:")
    user_content = "\n".join(blocks)
    chat = [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": user_content}]
    return qwen_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


def build_zero_shot_prompt(query_buggy):
    user_content = f"Fix this buggy Java method. Output the corrected method only:\n\n{query_buggy}"
    chat = [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": user_content}]
    return qwen_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def qwen_generate(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    inputs = qwen_tok(prompt, return_tensors="pt", truncation=True,
                      max_length=GEN_MAX_CTX_LEN).to(qwen_model.device)
    out = qwen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.0),
        temperature=temperature if temperature > 0.0 else None,
        pad_token_id=qwen_tok.pad_token_id,
        eos_token_id=qwen_tok.eos_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return qwen_tok.decode(gen, skip_special_tokens=True).strip()


# Strip markdown / leading prose so the raw code is left for metric computation
import re
def extract_code(text):
    m = re.search(r"```(?:java)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        return m.group(1).strip()
    for tag in ("```java", "```"):
        if text.startswith(tag):
            return text.replace(tag, "", 1).strip().rstrip("`").strip()
    return text.strip()


# Sanity check on a single example
sample_q = test_pairs[0]["buggy"]
sample_hits = retriever.retrieve(sample_q, top_k=TOP_K)
sample_prompt = build_few_shot_prompt(sample_q, sample_hits)
print(f"Prompt preview ({len(sample_prompt)} chars):\n")
print(sample_prompt[:800], "...")

Prompt preview (2645 chars):

<|im_start|>system
You are an expert Java developer fixing bugs in methods. Given a buggy Java method, respond with ONLY the corrected method. Do not add explanations, comments, or markdown — output raw Java code.<|im_end|>
<|im_start|>user
Here are examples of buggy Java methods and their fixes:

--- Example 1 ---
Buggy:
private boolean METHOD_1 ( ) { VAR_1 = null ; if ( VAR_2 . METHOD_2 ( ( ( VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) ) != null ? VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) : VAR_3 . METHOD_5 ( ) . METHOD_6 ( ) ) ) ) { execute ( VAR_2 ) ; return true ; } else { VAR_2 = null ; return METHOD_7 ( ) ; } }


Fixed:
private boolean METHOD_1 ( ) { VAR_1 = null ; if ( ( VAR_2 . METHOD_2 ( ( ( VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) ) != null ? VAR_3 . METHOD_3 ( ) . METHOD_4 ( 0 ) : VAR_3 . M ...


---
---

# Task 4: Inference on the Test Set

---
---

## 4.1) Load Test Set (Same as Notebook 2)

In [15]:
test_split = test_pairs
if TEST_SUBSET_SIZE is not None:
    test_split = test_split[:min(TEST_SUBSET_SIZE, len(test_split))]
print(f"Test instances: {len(test_split):,}")

Test instances: 500


## 4.2) RAG (3-shot) Generation

In [16]:
test_subset = test_pairs[:TEST_SUBSET_SIZE]

rag_refs, rag_hyps = [], []
for i, row in enumerate(tqdm(test_subset, desc="RAG 3-shot")):
    hits = retriever.retrieve(row["buggy"], top_k=TOP_K)
    prompt = build_few_shot_prompt(row["buggy"], hits)
    raw = qwen_generate(prompt)
    rag_hyps.append(extract_code(raw))
    rag_refs.append(row["fixed"])
    if (i + 1) % 20 == 0 and DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

with open(f"{RESULTS_DIR}/preds_RAG.json", "w") as f:
    json.dump({"refs": rag_refs, "hyps": rag_hyps}, f)
print(f"✓ Saved {len(rag_hyps)} RAG predictions")

RAG 3-shot:   0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
RAG 3-shot: 100%|██████████| 500/500 [52:23<00:00,  6.29s/it]

✓ Saved 500 RAG predictions


## 4.3) Zero-shot Generation

In [17]:
zs_refs, zs_hyps = [], []
for i, row in enumerate(tqdm(test_subset, desc="Zero-shot")):
    prompt = build_zero_shot_prompt(row["buggy"])
    raw = qwen_generate(prompt)
    zs_hyps.append(extract_code(raw))
    zs_refs.append(row["fixed"])
    if (i + 1) % 20 == 0 and DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

with open(f"{RESULTS_DIR}/preds_ZeroShot.json", "w") as f:
    json.dump({"refs": zs_refs, "hyps": zs_hyps}, f)
print(f"✓ Saved {len(zs_hyps)} zero-shot predictions")

Zero-shot: 100%|██████████| 500/500 [41:08<00:00,  4.94s/it]

✓ Saved 500 zero-shot predictions


---
---

# Task 5: Evaluation — All Four Configurations

---
---

In [8]:
!pip install -q git+https://github.com/k4black/codebleu.git@0.7.1
!pip install -q "tree-sitter==0.23.2" "tree-sitter-java==0.23.5"
from codebleu import calc_codebleu

def exact_match(refs, hyps):
    norm = lambda s: ' '.join(s.split())
    return sum(norm(r) == norm(h) for r, h in zip(refs, hyps)) / len(refs)

def compute_metrics(refs, hyps, tag):
    cb = calc_codebleu(refs, hyps, lang="java", weights=(0.25, 0.25, 0.25, 0.25))
    return {
        "tag": tag,
        "n": len(refs),
        "codebleu": round(cb["codebleu"] * 100, 2),
        "ngram_match": round(cb["ngram_match_score"] * 100, 2),
        "weighted_ngram": round(cb["weighted_ngram_match_score"] * 100, 2),
        "syntax_match": round(cb["syntax_match_score"] * 100, 2),
        "dataflow_match": round(cb["dataflow_match_score"] * 100, 2),
        "exact_match": round(exact_match(refs, hyps) * 100, 2),
    }

metrics_rag = compute_metrics(rag_refs, rag_hyps, "RAG 3-shot (Qwen 1.5B)")
metrics_zs  = compute_metrics(zs_refs,  zs_hyps,  "Zero-shot (Qwen 1.5B)")

with open(f"{RESULTS_DIR}/metrics_RAG.json", "w") as f: json.dump(metrics_rag, f, indent=2)
with open(f"{RESULTS_DIR}/metrics_ZeroShot.json", "w") as f: json.dump(metrics_zs, f, indent=2)

print(json.dumps(metrics_rag, indent=2))
print(json.dumps(metrics_zs,  indent=2))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
codebleu 0.7.1 requires tree-sitter<0.23.0,>=0.22.0, but you have tree-sitter 0.23.2 which is incompatible.
{
  "tag": "RAG 3-shot (Qwen 1.5B)",
  "n": 500,
  "codebleu": 57.39,
  "ngram_match": 34.0,
  "weighted_ngram": 39.25,
  "syntax_match": 73.45,
  "dataflow_match": 82.85,
  "exact_match": 1.4
}
{
  "tag": "Zero-shot (Qwen 1.5B)",
  "n": 500,
  "codebleu": 37.83,
  "ngram_match": 3.39,
  "weighted_ngram": 6.01,
  "syntax_match": 63.88,
  "dataflow_match": 78.04,
  "exact_match": 0.0
}


## 5.1) Final Comparison — All Four Configurations

In [9]:
# Load the two T5 metrics written by Notebook 2
with open(f"{RESULTS_DIR}/metrics_A.json") as f: metrics_A = json.load(f)
with open(f"{RESULTS_DIR}/metrics_B.json") as f: metrics_B = json.load(f)

all_metrics = [metrics_A, metrics_B, metrics_rag, metrics_zs]

print("=" * 88)
print(f"{'Configuration':<42} | {'N':>5} | {'CodeBLEU':>8} | {'EM (%)':>7}")
print("=" * 88)
for m in all_metrics:
    print(f"{m['tag']:<42} | {m['n']:>5} | {m['codebleu']:>8.2f} | {m['exact_match']:>7.2f}")
print("=" * 88)

print("\nCodeBLEU component breakdown:")
print(f"{'':<42} | {'ngram':>6} | {'w.ngram':>7} | {'syntax':>6} | {'dflow':>6}")
for m in all_metrics:
    print(f"  {m['tag']:<40} | {m['ngram_match']:>6.2f} | {m['weighted_ngram']:>7.2f} | "
          f"{m['syntax_match']:>6.2f} | {m['dataflow_match']:>6.2f}")

with open(f"{RESULTS_DIR}/final_comparison.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"\n✓ Final comparison saved → {RESULTS_DIR}/final_comparison.json")

Configuration                              |     N | CodeBLEU |  EM (%)
Pipeline A (pre-trained + fine-tuned)      |  6545 |    64.51 |    0.08
Pipeline B (fine-tuned only)               |  6545 |    86.82 |    2.32
RAG 3-shot (Qwen 1.5B)                     |   500 |    57.39 |    1.40
Zero-shot (Qwen 1.5B)                      |   500 |    37.83 |    0.00

CodeBLEU component breakdown:
                                           |  ngram | w.ngram | syntax |  dflow
  Pipeline A (pre-trained + fine-tuned)    |  67.73 |   67.99 |  61.00 |  61.30
  Pipeline B (fine-tuned only)             |  90.50 |   90.51 |  82.46 |  83.82
  RAG 3-shot (Qwen 1.5B)                   |  34.00 |   39.25 |  73.45 |  82.85
  Zero-shot (Qwen 1.5B)                    |   3.39 |    6.01 |  63.88 |  78.04

✓ Final comparison saved → /content/drive/MyDrive/W&M/GenAI/Assignment_3/results/final_comparison.json


## 5.2) Notes for the Write-up

- **Pre-training payoff** — compare A vs B: does the span-corruption objective on 50K Java methods yield better bug-fixing performance, or is 3 epochs × 50K too little signal?
- **Retrieval quality** — compare RAG vs Zero-shot: does supplying 3 similar buggy→fixed examples lift Qwen's CodeBLEU?
- **Paradigm comparison** — T5-small (~60M, task-specific) vs Qwen 1.5B (general + retrieved context):
  - Compute cost: A/B inference is ms/sample on one GPU; RAG pays 25× more per call plus a retrieval hop.
  - Data cost: A/B require 52K labelled pairs; RAG reuses them as a KB instead of training signal.
  - Flexibility: change the domain → A/B need retraining; RAG just needs a new KB.
- **Deployment angle** — a linter-plugin vendor probably wants a fine-tuned T5; a research lab doing a one-off study is fine with RAG.